# Reading Molecules as Graphs: the Math and Data Behind a ¹³C NMR Graph Network
**Series B: Deep Learning for Molecules, Post 3** &nbsp;|&nbsp; companion *primer* notebook

This is the lightweight companion to the full tutorial notebook `GCN_NMR_Tutorial.ipynb` (RDKit, PyTorch Geometric, a GPU, and the NMRShiftDB data file).
**This primer needs none of those.** It runs on a CPU in a few seconds, with NumPy only, and lets you check the ideas and numbers of the post.

| Section | What you verify |
|---|---|
| 1 | one graph convolution by hand, against the numbers the full tutorial prints |
| 2 | what a carbon "sees" after $k$ layers, on the caffeine graph |
| 3 | why very deep graph networks blur atoms together (over-smoothing) |
| 4 | batching many molecules as one disconnected graph |
| 5 | where the 207,071 parameters live, and how depth changes them |
| 6 | a synthetic experiment: how accuracy depends on how far the model can see |
| 7 | reading the saved test metrics (R², tails, the baseline) |
| 8 | **optional, needs the data file:** the dataset facts, train/test overlap, and a noise floor from replicate spectra |

**How to use it:** run top to bottom (**Runtime → Run all**), try the **Stop and predict** cells, then do the exercises.
Numbers about the *trained network* (learning curve, test metrics) are typed in from the saved run of the full tutorial.

## 0. Setup

In [ ]:
import os, math
import numpy as np, pandas as pd, matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")

# Section 8 is optional: it needs the NMRShiftDB pickle of the full tutorial (30.7 MB).
DATA_NAMES = ["graph_conv_many_nuc_pipeline.data.13C.nmrshiftdb_hconfspcl_nmrshiftdb.aromatic.64.0.mol_dict.pickle",
              "graph_conv_many_nuc_pipeline_data_13C_nmrshiftdb_hconfspcl_nmrshiftdb_aromatic_64_0_mol_dict.pickle"]
DATA_PATH = next((p for p in DATA_NAMES if os.path.exists(p)), None)
print("data file found:", DATA_PATH if DATA_PATH else "no (Section 8 will be skipped; everything else runs without it)")

## 1. One graph convolution by hand
The layer used in the full tutorial (Kipf and Welling) is
$$H^{(l+1)}=\sigma\big(\hat A\,H^{(l)}W^{(l)}\big),\qquad \hat A=\tilde D^{-1/2}\tilde A\,\tilde D^{-1/2},\quad \tilde A=A+I.$$
Toy molecule: three atoms in a chain, $0-1-2$, one feature each, $X=[1,2,4]$. With $W=1$ the layer is pure neighbourhood averaging, $\hat AX$.
**Stop and predict:** the full tutorial prints `[1.31649649, 2.7079079, 2.8164964]`. Which entries of $\hat A$ are 0.5, 0.408 and 0.333?

In [ ]:
A = np.zeros((3, 3))
for i, j in [(0, 1), (1, 2)]:
    A[i, j] = A[j, i] = 1                          # bonds 0-1 and 1-2
A_tilde = A + np.eye(3)                            # add self-loops
deg = A_tilde.sum(1)                               # [2, 3, 2]
A_hat = A_tilde / np.sqrt(deg)[:, None] / np.sqrt(deg)[None, :]
X = np.array([1.0, 2.0, 4.0])

print("degrees (with self-loops):", deg)
print("A_hat =\n", A_hat.round(3))
print("A_hat @ X =", (A_hat @ X).round(6), "  (full tutorial: 1.316496, 2.707908, 2.816496)")
print("\nentry (0,0) = 1/deg0 =", 1 / deg[0], "| entry (0,1) = 1/sqrt(deg0*deg1) =", round(1 / math.sqrt(deg[0] * deg[1]), 3),
      "| entry (1,1) = 1/deg1 =", round(1 / deg[1], 3))

## 2. What does a carbon see after $k$ layers?
$(\hat A^k)_{ij}\neq0$ exactly when atom $j$ is within $k$ bonds of atom $i$ (the self-loops let a walk stand still). So **$k$ stacked layers give each atom a field of view of $k$ bonds**.
We use the heavy-atom graph of **caffeine**, `CN1C=NC2=C1C(=O)N(C(=O)N2C)C` (the full tutorial adds hydrogens, which only extend the graph at its edges). The full network has **6** graph convolutions.
**Stop and predict:** how many layers does the methyl carbon `C0` need to see every atom of the molecule?

In [ ]:
atoms = ["C", "N", "C", "N", "C", "C", "C", "O", "N", "C", "O", "N", "C", "C"]
bonds = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,1),(5,6),(6,7),(6,8),(8,9),(9,10),(9,11),(11,4),(11,12),(8,13)]
n = len(atoms)
Ac = np.zeros((n, n))
for i, j in bonds: Ac[i, j] = Ac[j, i] = 1

dist = np.full((n, n), -1)                          # shortest path length in bonds (breadth-first search)
for s in range(n):
    dist[s, s] = 0; frontier = [s]; d = 0
    while frontier:
        d += 1; nxt = []
        for u in frontier:
            for v in np.where(Ac[u] > 0)[0]:
                if dist[s, v] < 0: dist[s, v] = d; nxt.append(v)
        frontier = nxt

carbons = [i for i, a in enumerate(atoms) if a == "C"]
print("carbons:", carbons, "| diameter of the graph:", dist.max(), "bonds")
table = pd.DataFrame({f"C{c}": [int((dist[c] <= k).sum()) for k in range(1, 7)] for c in carbons}, index=[f"within {k} bonds" for k in range(1, 7)])
print(table.to_string())
ecc = {c: int(dist[c].max()) for c in carbons}
print("\nlayers needed to see the whole molecule:", ecc)

# the receptive field IS the nonzero pattern of A_hat^k
At_c = Ac + np.eye(n); dg = At_c.sum(1); Ah_c = At_c / np.sqrt(dg)[:, None] / np.sqrt(dg)[None, :]
for k in (1, 3, 6):
    nz = (np.linalg.matrix_power(Ah_c, k) > 1e-12)
    assert (nz == (dist <= k)).all()
print("nonzero pattern of A_hat^k equals 'within k bonds' for k = 1, 3, 6: True")

## 3. Why very deep graph networks blur atoms together
$\hat A$ is symmetric with largest eigenvalue exactly 1 (eigenvector $\propto\sqrt{\tilde d_i}$). Repeated multiplication therefore drives every feature column toward that one direction: atoms become **proportional to each other**, and the differences between them shrink at least as fast as $\lambda^k$, where $\lambda$ is the largest remaining $|$eigenvalue$|$. (The real network has weights, ReLUs, BatchNorm and skip connections, so this is the linear caricature of *over-smoothing*.)

In [ ]:
w = np.linalg.eigvalsh(Ah_c)[::-1]
lam = max(abs(w[1]), abs(w[-1]))
print("largest eigenvalues of A_hat:", w[:3].round(4), "| smallest:", w[-1].round(3), "| lambda =", lam.round(4))

v1 = np.sqrt(dg); v1 /= np.linalg.norm(v1)          # dominant eigenvector
def spread(M):                                      # size of the part of M that is NOT along v1: how different the atoms still are
    return np.linalg.norm(M - np.outer(v1, v1 @ M))

rng = np.random.default_rng(0)
X5 = rng.standard_normal((n, 5)); base = spread(X5)
ks = np.array([0, 1, 2, 3, 6, 12, 24])
ratio = np.array([spread(np.linalg.matrix_power(Ah_c, k) @ X5) / base for k in ks])
print(pd.DataFrame({"layers k": ks, "spread / initial": ratio.round(3), "bound lambda^k": (lam ** ks).round(3)}).to_string(index=False))
assert (ratio <= lam ** ks + 1e-9).all()

## 4. Batching many molecules as one disconnected graph
PyTorch Geometric merges a batch into one big graph: atoms are stacked, each molecule's atom numbers are **shifted**, and no edge crosses molecules. `ptr` marks where each molecule starts, and `batch[i]` says which molecule atom $i$ belongs to.
Here: ethanol *with* hydrogens (the edge list printed in the full tutorial) and caffeine (heavy atoms).

In [ ]:
edge_ethanol = np.array([[0,1,1,2,0,3,0,4,0,5,1,6,1,7,2,8],
                         [1,0,2,1,3,0,4,0,5,0,6,1,7,1,8,2]])               # as printed by the full tutorial: 9 atoms, 16 directed edges
edge_caf = np.array([[i for i, j in bonds] + [j for i, j in bonds], [j for i, j in bonds] + [i for i, j in bonds]])
sizes = [9, 14]
ptr = np.concatenate([[0], np.cumsum(sizes)])                                # molecule start positions: [0, 9, 23]
batch_vec = np.repeat(np.arange(len(sizes)), sizes)                          # which molecule each atom belongs to
edge_batch = np.concatenate([edge_ethanol, edge_caf + ptr[1]], axis=1)       # shift caffeine's atom numbers by 9

print("ptr:", ptr.tolist(), "| batch vector:", batch_vec.tolist())
print("directed edges:", edge_ethanol.shape[1], "+", edge_caf.shape[1], "=", edge_batch.shape[1], "(each bond is stored twice)")
src, dst = edge_batch
assert (batch_vec[src] == batch_vec[dst]).all()
print("every edge stays inside one molecule: True")

local_idx = np.arange(sum(sizes)) - ptr[batch_vec]                           # atom index WITHIN its own molecule
print("local atom index of the first atom of each molecule:", [int(local_idx[ptr[m]]) for m in range(2)],
      "| carbon 12 of caffeine is global atom", ptr[1] + 12, "-> local", int(local_idx[ptr[1] + 12]))

A_big = np.zeros((sum(sizes), sum(sizes))); A_big[src, dst] = 1
assert not A_big[:9, 9:].any() and not A_big[9:, :9].any()
print("the big adjacency matrix is block-diagonal: True")

## 5. Where do the 207,071 parameters live?
The network: input BatchNorm (47) $\to$ `GCNConv` 47 to 128 $\to$ 5 × `ResGraph` (BatchNorm + `GCNConv` 128 to 128) $\to$ BatchNorm, `Linear` 128 to 128 $\to$ 3 × `ResBlock` (BatchNorm + 2 `Linear`) $\to$ BatchNorm, `Linear` 128 to 1.
A `Linear` $m\to n$ has $mn+n$ parameters, a BatchNorm over $n$ features has $2n$, and a `GCNConv` $m\to n$ has $mn+n$ (a weight matrix and a bias).

In [ ]:
lin = lambda m, n: m * n + n
bn  = lambda n: 2 * n
gcn = lambda m, n: m * n + n

def total_params(n_graph=5, n_res=3, h=128, f=47):
    parts = {"input BatchNorm":               bn(f),
             "first GCNConv (47 -> 128)":      gcn(f, h),
             f"{n_graph} x ResGraph":          n_graph * (gcn(h, h) + bn(h)),
             "BatchNorm after the graph part": bn(h),
             "bridge Linear":                  lin(h, h),
             f"{n_res} x ResBlock":            n_res * (2 * lin(h, h) + bn(h)),
             "final BatchNorm":                bn(h),
             "output Linear (128 -> 1)":       lin(h, 1)}
    return parts

parts = total_params(); tot = sum(parts.values())
for k, v in parts.items(): print(f"{k:34s} {v:>10,d}  {100 * v / tot:5.1f}%")
print(f"{'TOTAL':34s} {tot:>10,d}   (the full tutorial prints 207,071: {tot == 207_071})")

print("\nDepth ablation (Exercise 1 of the full tutorial): parameters and receptive field")
rows = [(nb, sum(total_params(nb).values()), nb + 1) for nb in (0, 1, 2, 5, 8)]
print(pd.DataFrame(rows, columns=["n_graph_blocks", "parameters", "bonds seen (layers)"]).to_string(index=False))

## 6. A synthetic experiment: how far must the model see?
Real ¹³C shifts depend on the neighbourhood out to a few bonds (the idea behind classical HOSE codes). Here is a toy where we know the truth. For random branched carbon skeletons ("alkane-like trees"), generate a synthetic shift for every carbon from an **additive rule** with increments of the kind used in classical alkane shift estimation, $\delta=-2.3+9.1\,n_1+9.4\,n_2-2.5\,n_3+0.3\,n_4$ (+ noise, std 0.5 ppm), where $n_k$ is the number of carbons exactly $k$ bonds away. The exact increments are unimportant; this is a *synthetic* illustration, not chemistry.
Then fit a linear model that can only use the counts out to radius $r$. (This stands in for a network of $r$ layers; it is not a trained GCN.)
**Stop and predict:** which radius brings the error down to the noise floor?

In [ ]:
def random_tree(rng, n, maxdeg=4):
    adj = [[] for _ in range(n)]
    for v in range(1, n):
        while True:
            u = int(rng.integers(0, v))
            if len(adj[u]) < maxdeg: break
        adj[u].append(v); adj[v].append(u)
    return adj

def bfs(adj, s):
    d = {s: 0}; fr = [s]
    while fr:
        nx = []
        for u in fr:
            for v in adj[u]:
                if v not in d: d[v] = d[u] + 1; nx.append(v)
        fr = nx
    return d

TRUE = np.array([-2.3, 9.1, 9.4, -2.5, 0.3]); NOISE = 0.5
rng = np.random.default_rng(0)
feats, shifts = [], []
for _ in range(1500):
    nn = int(rng.integers(8, 19)); adj = random_tree(rng, nn)
    for c in range(nn):
        cnt = np.zeros(5)
        for v, dd in bfs(adj, c).items():
            if 1 <= dd <= 4: cnt[dd] += 1
        feats.append(cnt[1:]); shifts.append(TRUE[0] + TRUE[1:] @ cnt[1:] + rng.normal(0, NOISE))
Xt, yt = np.array(feats), np.array(shifts)
N = len(yt); tr = np.arange(N) < int(0.8 * N); te = ~tr
print(f"{N:,} synthetic carbons | shifts {yt.min():.0f} to {yt.max():.0f} ppm | baseline (mean) MAE {np.abs(yt[te] - yt[tr].mean()).mean():.2f}")
print(f"noise floor for Gaussian noise of std {NOISE}: MAE = {NOISE * math.sqrt(2 / math.pi):.2f}\n")

res = []
for r in range(0, 5):
    F = np.c_[np.ones(N), Xt[:, :r]]
    beta = np.linalg.lstsq(F[tr], yt[tr], rcond=None)[0]; pred = F[te] @ beta
    res.append((r, np.abs(pred - yt[te]).mean(), np.sqrt(((pred - yt[te]) ** 2).mean()), np.round(beta, 2)))
    print(f"radius {r}: MAE {res[-1][1]:.2f}  RMSE {res[-1][2]:.2f}   fitted increments {res[-1][3]}")

plt.figure(figsize=(6, 3.4))
plt.semilogy([r[0] for r in res], [r[1] for r in res], "o-", color=COLORS["blue"])
plt.axhline(NOISE * math.sqrt(2 / math.pi), color=COLORS["gray"], ls="--"); plt.text(0.05, 0.43, "noise floor", color=COLORS["gray"], fontsize=8)
plt.xlabel("radius r: bonds the model can see"); plt.ylabel("test MAE (ppm, log scale)"); plt.xticks(range(5)); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 7. Reading the saved test metrics
Typed in from the saved run of the full tutorial (T4 GPU, 100 epochs, 2.8 minutes). The model was selected on the validation set; the test set was used once.

In [ ]:
metrics = pd.DataFrame([("validation", 17560, 2.132, 3.189, 1.449, 0.996),
                        ("TEST",       44560, 2.133, 3.288, 1.440, 0.996),
                        ("baseline: predict the mean", 44560, 46.310, 51.207, 41.780, -0.003)],
                       columns=["set", "carbons", "MAE", "RMSE", "median |err|", "R2"])
print(metrics.to_string(index=False))

# 1) is R2 consistent with the RMSE?  R2 = 1 - MSE/Var.  The baseline has R2 = -0.003, so Var = MSE_baseline / 1.003.
var_test = 51.207 ** 2 / 1.003
print(f"\nR2 recomputed from the test RMSE: {1 - 3.288 ** 2 / var_test:.4f}  (reported 0.996)")

# 2) tails: for Gaussian errors MAE/RMSE = 0.798 and median/MAE = 0.845
print(f"MAE/RMSE      : test {2.133 / 3.288:.3f}, validation {2.132 / 3.189:.3f}   (Gaussian errors: {math.sqrt(2 / math.pi):.3f})")
print(f"median/MAE    : test {1.440 / 2.133:.3f}, validation {1.449 / 2.132:.3f}   (Gaussian errors: {0.6745 / math.sqrt(2 / math.pi):.3f})")
print("-> both ratios are well below the Gaussian values: a few large errors dominate the squared error (heavy tails).")
print(f"\nimprovement over the baseline: MAE x{46.310 / 2.133:.1f} smaller, RMSE x{51.207 / 3.288:.1f} smaller")

In [ ]:
epochs = [1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
train_mae = [44.09, 4.05, 3.37, 3.01, 2.97, 3.07, 2.83, 2.67, 2.82, 2.49, 2.51]
val_mae   = [33.53, 4.89, 3.34, 3.59, 3.48, 2.90, 2.69, 2.19, 2.30, 2.76, 2.99]
plt.figure(figsize=(7, 3.6))
plt.semilogy(epochs, train_mae, "o-", color=COLORS["blue"], label="train (running)")
plt.semilogy(epochs, val_mae,   "s-", color=COLORS["orange"], label="validation")
plt.scatter([98], [2.13], color=COLORS["red"], zorder=5, label="best validation: 2.13 (epoch 98)")
plt.xlabel("epoch"); plt.ylabel("MAE (ppm, log scale)"); plt.grid(alpha=0.3); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

late = np.array(val_mae[6:])                      # logged epochs 60 to 100
print(f"validation MAE at the logged epochs 60..100: {late.tolist()} -> range {late.max() - late.min():.2f} ppm, std {late.std(ddof=1):.2f} ppm")
print("The curve is noisy from epoch to epoch (constant learning rate, BatchNorm), so 'best epoch' is a minimum of a noisy series.")

## 8. Optional: the dataset itself (needs the data file)
The full tutorial loads the data with `pandas.read_pickle`. **A pickle file can execute arbitrary code when loaded.** Instead of trusting the file, we *read what it contains without running anything*: `pickletools` lists every Python callable the file would invoke, and then a **whitelist-only unpickler** maps each of those names to an inert stand-in class, so nothing but our own stubs can ever run. This also means we need neither RDKit nor a matching pandas version. (We cannot look *inside* the molecules this way, but we can read everything else: ids and the experimental shifts.)

In [ ]:
HAVE_DATA = DATA_PATH is not None
if not HAVE_DATA:
    try:
        from google.colab import files
        print("Optional: upload the dataset pickle to run this section (or cancel to skip it).")
        up = files.upload()
        if up: DATA_PATH = next(iter(up)); HAVE_DATA = True
    except Exception:
        print("No data file: skipping Section 8.")

if HAVE_DATA:
    import pickle, pickletools, collections
    # --- step 1: list, WITHOUT executing anything, every global the pickle would import or call -----------------------------
    raw = open(DATA_PATH, "rb").read()
    strings, memo, last, n_memo, found = [], {}, None, 0, collections.Counter()
    for op, arg, pos in pickletools.genops(raw):
        if op.name in ("SHORT_BINUNICODE", "BINUNICODE", "BINUNICODE8"): strings.append(arg); last = arg
        elif op.name == "MEMOIZE": memo[n_memo] = last; n_memo += 1
        elif op.name in ("BINPUT", "LONG_BINPUT"): memo[arg] = last
        elif op.name in ("BINGET", "LONG_BINGET"): last = memo.get(arg); strings.append(last)
        elif op.name == "GLOBAL": found[arg] += 1
        elif op.name == "STACK_GLOBAL" and len(strings) >= 2: found[f"{strings[-2]} {strings[-1]}"] += 1
    print(f"{os.path.getsize(DATA_PATH) / 1e6:.1f} MB; the pickle references only these callables:")
    for k, v in found.most_common(): print(f"   {v:>3} x {k}")

In [ ]:
if HAVE_DATA:
    try:
        from numpy._core.multiarray import _reconstruct as _np_reconstruct
    except ImportError:
        from numpy.core.multiarray import _reconstruct as _np_reconstruct

    class Stub:                                     # inert stand-in: stores whatever it is given, runs nothing
        def __init__(self, *a, **k): self._args, self._kw = a, k
        def __setstate__(self, s): self._state = s
    class DataFrame(Stub): pass
    class BlockManager(Stub): pass
    class Index(Stub): pass
    class Int64Index(Stub): pass
    class Mol(Stub): pass                           # an RDKit molecule becomes a holder of its serialised bytes
    def _new_Index(cls, d): o = cls.__new__(cls); o._kw = d; return o

    ALLOWED = {("pandas.core.frame", "DataFrame"): DataFrame, ("pandas.core.internals", "BlockManager"): BlockManager,
               ("pandas.core.indexes.base", "_new_Index"): _new_Index, ("pandas.core.indexes.base", "Index"): Index,
               ("pandas.core.indexes.numeric", "Int64Index"): Int64Index, ("rdkit.Chem.rdchem", "Mol"): Mol,
               ("builtins", "slice"): slice, ("numpy", "ndarray"): np.ndarray, ("numpy", "dtype"): np.dtype,
               ("numpy.core.multiarray", "_reconstruct"): _np_reconstruct}

    class SafeUnpickler(pickle.Unpickler):
        def find_class(self, module, name):
            if (module, name) not in ALLOWED:
                raise pickle.UnpicklingError(f"blocked: {module}.{name}  (do not load this file)")
            return ALLOWED[(module, name)]

    with open(DATA_PATH, "rb") as fh:
        data = SafeUnpickler(fh).load()

    def columns(df):                                # pull the columns out of the stub's block structure
        axes, blocks, names, _ = df._state["_data"]._state
        out = {}
        for arr, it in zip(blocks, names):
            for j, nm in enumerate(list(it._kw["data"])): out[nm] = arr[j]
        return out

    tables = {k: columns(data[k]) for k in ("train_df", "test_df")}
    shifts = {k: [dict(v[0]) for v in tables[k]["value"]] for k in tables}      # one {atom index: ppm} dict per spectrum
    print("keys:", list(data.keys()), "| MAX_N:", data["MAX_N"], "| nuclei:", data["tgt_nucs"])
    print("columns:", list(tables["train_df"].keys()))

In [ ]:
if HAVE_DATA:
    flat = {k: np.array([s for dd in shifts[k] for s in dd.values()]) for k in shifts}
    rows = []
    for k, label in (("train_df", "training"), ("test_df", "test")):
        ids = tables[k]["molecule_id"]; u, cnt = np.unique(ids, return_counts=True); a = flat[k]
        rows.append((label, len(ids), len(u), int((cnt > 1).sum()), len(a), f"{a.min():.1f} to {a.max():.1f}", f"{a.mean():.1f} +/- {a.std():.1f}"))
    print(pd.DataFrame(rows, columns=["table", "spectra", "unique molecule ids", "molecules with >1 spectrum", "labelled carbons", "shift range (ppm)", "mean +/- std"]).to_string(index=False))

    tr_ids, te_ids = set(tables["train_df"]["molecule_id"]), set(tables["test_df"]["molecule_id"])
    tr_bytes = {m._args[0] for m in tables["train_df"]["rdmol"]}
    print(f"\nmolecule ids in BOTH tables: {len(tr_ids & te_ids)} | test molecules whose exact serialisation also occurs in training: "
          f"{sum(m._args[0] in tr_bytes for m in tables['test_df']['rdmol'])}")
    print("(this checks identical ids and identical stored molecules; it cannot detect the same structure under a different id)")

    a = flat["train_df"]
    print("\nshare of training carbons by region:", {f"{lo}-{hi}": f"{100 * ((a >= lo) & (a < hi)).mean():.1f}%" for lo, hi in ((-50, 50), (50, 100), (100, 160), (160, 400))})
    print(f"labelled carbons per spectrum: mean {np.mean([len(d) for d in shifts['train_df']]):.2f}")
    at = flat["test_df"]
    print(f"always predicting the training mean ({a.mean():.1f}) on ALL test carbons: MAE {np.abs(at - a.mean()).mean():.2f}, RMSE {math.sqrt(((at - a.mean()) ** 2).mean()):.2f}"
          "   (the full tutorial's 46.31 / 51.21 are for its complete-spectra subset)")

    print("\nthe two entries printed in the full tutorial:")
    print(" training entry 13:", dict(sorted(shifts["train_df"][13].items())))
    print(" test row 20       :", dict(sorted(shifts["test_df"][20].items())))

### How noisy are the labels? Replicate spectra
Some molecules have several spectra (same `molecule_id`, same stored molecule, hence the same atom numbering). If the same carbon is reported with different shifts in two spectra, that disagreement tells us how noisy *experimental labels* are, which sets a floor under any model's error.
Suppose two reports of a carbon are $f+\varepsilon_1$ and $f+\varepsilon_2$ with independent, zero-mean errors of the same size. Then $E[(\varepsilon_1-\varepsilon_2)^2]=2\sigma^2$, so the label-noise variance is $\sigma^2=\tfrac12E[d^2]$ for the replicate differences $d$. A model that predicted $f$ perfectly would still have mean squared error $\sigma^2$ against a single measurement. For the absolute error (no assumption on the shape of the errors): $\tfrac12E|d|\le E|\varepsilon|\le E|d|$.

In [ ]:
if HAVE_DATA:
    def replicate_differences(k):
        ids = tables[k]["molecule_id"]; mol = [m._args[0] for m in tables[k]["rdmol"]]
        groups = collections.defaultdict(list)
        for i, m in enumerate(ids): groups[m].append(i)
        d = []
        for g in groups.values():
            for x in range(len(g)):
                for y in range(x + 1, len(g)):
                    i, j = g[x], g[y]
                    if mol[i] == mol[j]:                               # same stored molecule -> same atom numbering
                        d += [shifts[k][i][c] - shifts[k][j][c] for c in set(shifts[k][i]) & set(shifts[k][j])]
        return np.array(d)

    rep = {k: replicate_differences(k) for k in tables}
    rows = []
    for k, label in (("train_df", "training"), ("test_df", "test")):
        d = np.abs(rep[k]); rows.append((label, len(d), d.mean(), np.median(d), np.percentile(d, 90), 100 * (d > 5).mean(), math.sqrt((rep[k] ** 2).mean())))
    print(pd.DataFrame(rows, columns=["replicates", "carbon pairs", "mean |d|", "median |d|", "90th pct", "% above 5 ppm", "RMS d"]).round(2).to_string(index=False))

    d = rep["test_df"]; sigma2 = (d ** 2).mean() / 2
    print(f"\nlabel-noise variance sigma^2 = E[d^2]/2 = {sigma2:.2f} ppm^2  ->  even a perfect model has RMSE >= {math.sqrt(sigma2):.2f} ppm")
    print(f"label-noise floor for the MAE lies between {np.abs(d).mean() / 2:.2f} and {np.abs(d).mean():.2f} ppm")
    print(f"the full tutorial's test MSE is 3.288^2 = {3.288 ** 2:.2f} ppm^2: label noise can explain at most {100 * sigma2 / 3.288 ** 2:.0f}% of it;"
          f" the rest, {3.288 ** 2 - sigma2:.2f} ppm^2 (RMSE-equivalent {math.sqrt(3.288 ** 2 - sigma2):.2f} ppm), is model error")

    plt.figure(figsize=(7, 3.4))
    plt.hist(np.clip(np.abs(rep["train_df"]), 0, 20), bins=np.linspace(0, 20, 41), color=COLORS["blue"]); plt.yscale("log")
    plt.xlabel("|difference between two spectra of the same carbon| (ppm, clipped at 20)"); plt.ylabel("carbon pairs"); plt.tight_layout(); plt.show()

In [ ]:
# The two largest errors of the full tutorial's test molecule (row 20): predicted | experimental, from its saved output
pred = {9: 119.4, 11: 111.7}; true = {9: 111.2, 11: 120.9}
print("as assigned :", {a: f"{pred[a]} predicted vs {true[a]} experimental -> error {pred[a] - true[a]:+.1f}" for a in pred})
print("if swapped  :", {a: f"{pred[a]} vs {true[11 if a == 9 else 9]} -> error {pred[a] - true[11 if a == 9 else 9]:+.1f}" for a in pred})
print("The two predictions match the two experimental values almost perfectly, but on the wrong atoms: either the model mixed up two similar")
print("aromatic carbons, or the assignment in the database is the one that is swapped. Such swaps are one source of the heavy error tails.")
if HAVE_DATA:
    t20 = shifts["test_df"][20]
    assert abs(t20[9] - 111.2) < 1e-6 and abs(t20[11] - 120.9) < 1e-6
    print("(the experimental values above agree with the data file)")

## 9. Your turn
**Exercise 1 (graph convolution).** Change the toy features to $X=[1,0,0]$ and compute $\hat AX$ and $\hat A^2X$ by hand. After how many layers has the signal from atom 0 reached atom 2?

**Exercise 2 (receptive field).** For the heavy-atom caffeine graph, which carbon has the *smallest* field of view after 2 layers, and which the largest? How many layers does the full network need (with 6 convolutions, is it enough for every carbon)?

**Exercise 3 (parameters).** How many parameters does the network have with `HIDDEN = 64`? (Redo the derivation of Section 5 with $h=64$.)

**Exercise 4 (the toy radius experiment).** Change the noise to 2.0 ppm. At which radius does the MAE stop improving? Change the $\gamma$ increment from $-2.5$ to $-6$ and watch what happens at radius 2 and 3.

**Exercise 5 (metrics).** A classmate reports an MAE of 2.0 ppm and an RMSE of 2.5 ppm on the same test set. What does that tell you about their error tails, compared with the tutorial's 2.133 / 3.288?

**Exercise 6 (label noise).** With the replicate numbers of Section 8, what is the smallest RMSE a model can reach on this test set (if replicate errors are independent)? What would you conclude if a model reported an RMSE of 1.0 ppm?

In [ ]:
# Starters
# Exercise 1
X0 = np.array([1.0, 0.0, 0.0])
print("A_hat X =", (A_hat @ X0).round(3), "| A_hat^2 X =", (A_hat @ A_hat @ X0).round(3))

# Exercise 2
n_seen_2 = {f"C{c}": int((dist[c] <= 2).sum()) for c in carbons}
print("atoms seen after 2 layers:", n_seen_2)

# Exercise 3
print("parameters with HIDDEN = 64:", f"{sum(total_params(h=64).values()):,}")

# Exercise 5: for Gaussian errors with RMSE 2.5, what would the MAE be?
print("Gaussian errors with RMSE 2.5 would have MAE", round(2.5 * math.sqrt(2 / math.pi), 2), "-> the classmate's 2.0 is", "above" if 2.0 > 2.5 * math.sqrt(2 / math.pi) else "below", "that")

## 10. Self-check

In [ ]:
assert np.allclose(A_hat @ X, [1.31649658, 2.70790812, 2.81649658])
assert max(ecc.values()) == 6 and ecc[0] == 6 and int(dist.max()) == 6
assert tot == 207_071 and sum(total_params(0).values()) == 123_231
assert (ratio <= lam ** ks + 1e-9).all() and abs(lam - 0.9082) < 1e-3
assert (batch_vec[src] == batch_vec[dst]).all() and edge_batch.shape[1] == 16 + 30
assert res[3][1] < 0.6 and res[2][1] > 2 and abs(res[4][3][3] + 2.5) < 0.2
assert abs((1 - 3.288 ** 2 / var_test) - 0.9959) < 0.0005
if HAVE_DATA:
    assert [len(tables[k]["molecule_id"]) for k in tables] == [21_523, 5_390]
    assert [len(flat[k]) for k in flat] == [213_564, 53_279]
    assert len(tr_ids & te_ids) == 0 and (len(tr_ids), len(te_ids)) == (20_205, 5_055)
    assert abs(np.abs(rep["train_df"]).mean() - 0.98) < 0.01 and len(rep["train_df"]) == 12_053
print("All checks passed: you reproduced the numbers in the post.")